# publish_silver

**Source:** `03_publish/publish_silver.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Silver publish engine with append and merge modes."""

from __future__ import annotations

import re
import uuid

# Only allow simple identifiers as merge keys (column names or schema.table.col).
# This prevents SQL injection through metadata-driven merge_key values.
_SAFE_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


## Section 2: Define `_validate_identifier()` function with logic for processing

This cell handles: *Define `_validate_identifier()` function with logic for processing*


In [ ]:
def _validate_identifier(value: str, field_name: str) -> str:
    """Raise ValueError if *value* is not a safe SQL identifier."""
    if not _SAFE_IDENTIFIER.match(value):
        raise ValueError(
            f"{field_name} must be a valid SQL identifier (letters, digits, underscores, dots). "
            f"Got: {value!r}"
        )
    return value


## Section 3: Define `publish_append()` helper function

This cell handles: *Define `publish_append()` helper function*


In [ ]:
def publish_append(
    df,
    silver_table: str,
    partition_columns: list[str] | None = None,
    table_type: str = "managed",
    external_path: str | None = None,
):
    resolved_table_type = str(table_type).strip().lower() if table_type else "managed"
    if resolved_table_type not in {"managed", "external"}:
        raise ValueError("table_type must be 'managed' or 'external'")
    if resolved_table_type == "external" and not str(external_path or "").strip():
        raise ValueError("external_path is required when table_type=external")

    writer = df.write.mode("append").format("delta").option("mergeSchema", "true")
    if partition_columns:
        writer = writer.partitionBy(*partition_columns)
    if resolved_table_type == "external" and external_path:
        writer = writer.option("path", external_path)
    writer.saveAsTable(silver_table)


## Section 4: Define `publish_merge()` helper function

This cell handles: *Define `publish_merge()` helper function*


In [ ]:
def publish_merge(spark, df, silver_table: str, merge_key: str):
    publish_merge_with_target_policy(
        spark,
        df,
        silver_table,
        merge_key,
        table_type="managed",
        external_path=None,
    )


## Section 5: Define `publish_merge_with_target_policy()` function with logic for processing

This cell handles: *Define `publish_merge_with_target_policy()` function with logic for processing*


In [ ]:
def publish_merge_with_target_policy(
    spark,
    df,
    silver_table: str,
    merge_key: str,
    table_type: str = "managed",
    external_path: str | None = None,
):
    _validate_identifier(merge_key, "merge_key")
    _validate_identifier(silver_table, "silver_table")

    resolved_table_type = str(table_type).strip().lower() if table_type else "managed"
    if resolved_table_type not in {"managed", "external"}:
        raise ValueError("table_type must be 'managed' or 'external'")
    if resolved_table_type == "external" and not str(external_path or "").strip():
        raise ValueError("external_path is required when table_type=external")

    # Ensure merge target exists and respects metadata table policy.
    if not spark.catalog.tableExists(silver_table):
        init_writer = (
            df.limit(0)
            .write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
        )
        if resolved_table_type == "external" and external_path:
            init_writer = init_writer.option("path", external_path)
        init_writer.saveAsTable(silver_table)

    # Use a unique view name per merge operation to prevent cross-source
    # temp view collisions when multiple sources run in the same Spark session.
    view_name = f"_stg_silver_{uuid.uuid4().hex[:12]}"
    df.createOrReplaceTempView(view_name)
    try:
        spark.sql(
            f"""
            MERGE INTO {silver_table} AS tgt
            USING {view_name} AS src
            ON tgt.{merge_key} = src.{merge_key}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            """
        )
    finally:
        spark.catalog.dropTempView(view_name)


## Section 6: Define `apply_post_publish_actions()` helper function

This cell handles: *Define `apply_post_publish_actions()` helper function*


In [ ]:
def apply_post_publish_actions(spark, silver_table: str, zorder_columns: list[str] | None = None):
    _validate_identifier(silver_table, "silver_table")
    if zorder_columns:
        safe_cols = [_validate_identifier(c, "zorder_column") for c in zorder_columns]
        cols = ", ".join(safe_cols)
        spark.sql(f"OPTIMIZE {silver_table} ZORDER BY ({cols})")
    else:
        spark.sql(f"OPTIMIZE {silver_table}")
